### 用于评估显存占用和FLOPs

In [1]:
import torch
from fvcore.nn import FlopCountAnalysis
import numpy as np
import sys
import os

def measure_model_memory(model, dummy_input, device, num_warmup=3, num_runs=5):
    """测量模型显存占用"""
    if device != 'cuda':
        return {'peak_mb': 0}
    
    model.eval()
    
    torch.cuda.empty_cache()
    torch.cuda.synchronize()
    torch.cuda.reset_peak_memory_stats()
    
    # 预热
    with torch.no_grad():
        for _ in range(num_warmup):
            _ = model(dummy_input)
            torch.cuda.synchronize()
    
    # 测量峰值显存
    peak_memories = []
    for _ in range(num_runs):
        torch.cuda.empty_cache()
        torch.cuda.synchronize()
        torch.cuda.reset_peak_memory_stats()
        
        with torch.no_grad():
            _ = model(dummy_input)
            torch.cuda.synchronize()
        
        peak_mb = torch.cuda.max_memory_allocated() / (1024 * 1024)
        peak_memories.append(peak_mb)
    
    avg_peak_mb = sum(peak_memories) / len(peak_memories)
    
    return {'peak_mb': avg_peak_mb}

def compare_models_simple():
    """简化版模型对比 - 仅评估可训练参数、GFLOPs、Dice和HD95"""
    
    # Synapse数据集配置
    in_channels = 1
    num_classes = 14
    input_shape = (1, in_channels, 64, 128, 128)
    
    print(f"\n{'='*80}")
    print(f"模型效率对比 - Synapse数据集")
    print(f"{'='*80}")
    print(f"输入尺寸: {input_shape}")
    print(f"{'='*80}\n")
    
    device = 'cuda' if torch.cuda.is_available() else 'cpu'
    print(f"设备: {device}\n")
    
    models_info = []
    
    # 使用论文中报告的可训练参数量
    paper_trainable_params = {
        'LiteSegFormer3D': 1.73e6,
        'SegFormer3D': 4.51e6,
        'UNETR': 92.49e6
    }
    
    # 论文精度数据
    accuracy_data = {
        'LiteSegFormer3D': {'dice': 84.91, 'hd95': 9.41},
        'SegFormer3D': {'dice': 82.15, 'hd95': 20.00},
        'UNETR': {'dice': 79.56, 'hd95': 22.97}
    }
    
    # ==================== LiteSegFormer3D ====================
    print("分析 LiteSegFormer3D...")
    try:
        from lsf3d.network_architecture.litesegformer3d_synapse import LiteSegFormer3D
        
        model1 = LiteSegFormer3D(
            in_channels=in_channels,
            num_classes=num_classes,
            img_size=[64, 128, 128],
            sr_ratios=[8, 4, 2, 1],
            embed_dims=[32, 64, 128, 256],
            patch_kernel_size=[3, 3, 3, 3],
            patch_stride=[2, 2, 2, 2],
            patch_padding=[1, 1, 1, 1],
            mlp_ratios=[4, 4, 4, 4],
            num_heads=[1, 2, 4, 8],
            depths=[2, 2, 2, 2],
            decoder_head_embedding_dim=256,
            decoder_dropout=0.0,
            do_ds=False
        )
        
        if device == 'cuda':
            model1 = model1.cuda()
        model1.eval()
        
        dummy_input1 = torch.randn(input_shape).to(device)
        
        with torch.no_grad():
            output1 = model1(dummy_input1)
        
        flops1 = FlopCountAnalysis(model1, dummy_input1)
        gflops1 = flops1.total() / 1e9
        
        # 测量显存
        print(f"  测量显存 (warmup=3, runs=5)...")
        memory_info1 = measure_model_memory(model1, dummy_input1, device)
        print(f"  峰值显存: {memory_info1['peak_mb']:.2f} MB")
        
        models_info.append({
            'name': 'LiteSegFormer3D',
            'trainable_params': paper_trainable_params['LiteSegFormer3D'],
            'gflops': gflops1,
            'peak_mb': memory_info1['peak_mb'],
            'dice': accuracy_data['LiteSegFormer3D']['dice'],
            'hd95': accuracy_data['LiteSegFormer3D']['hd95']
        })
        
        del model1, dummy_input1
        if device == 'cuda':
            torch.cuda.empty_cache()
        print("✓ 完成\n")
    except Exception as e:
        print(f"✗ 失败: {e}\n")
    
    # ==================== UNETR ====================
    print("分析 UNETR...")
    try:
        from compare_model.unetr import UNETR
        
        model2 = UNETR(
            in_channels=in_channels,
            out_channels=num_classes,
            img_size=(64, 128, 128),
            feature_size=16,
            hidden_size=768,
            mlp_dim=3072,
            num_heads=12,
            pos_embed="perceptron",
            norm_name="instance",
            res_block=True,
            dropout_rate=0.0,
        )
        
        if device == 'cuda':
            model2 = model2.cuda()
        model2.eval()
        
        dummy_input2 = torch.randn(input_shape).to(device)
        
        with torch.no_grad():
            output2 = model2(dummy_input2)
        
        flops2 = FlopCountAnalysis(model2, dummy_input2)
        gflops2 = flops2.total() / 1e9
        
        # 测量显存
        print(f"  测量显存 (warmup=3, runs=5)...")
        memory_info2 = measure_model_memory(model2, dummy_input2, device)
        print(f"  峰值显存: {memory_info2['peak_mb']:.2f} MB")
        
        models_info.append({
            'name': 'UNETR',
            'trainable_params': paper_trainable_params['UNETR'],
            'gflops': gflops2,
            'peak_mb': memory_info2['peak_mb'],
            'dice': accuracy_data['UNETR']['dice'],
            'hd95': accuracy_data['UNETR']['hd95']
        })
        
        del model2, dummy_input2
        if device == 'cuda':
            torch.cuda.empty_cache()
        print("✓ 完成\n")
    except Exception as e:
        print(f"✗ 失败: {e}\n")
    
    # ==================== SegFormer3D ====================
    print("分析 SegFormer3D...")
    try:
        from compare_model.segformer3d import SegFormer3D as SF3D

        model3 = SF3D(
            in_channels=in_channels,
            sr_ratios=[8, 4, 2, 1],
            embed_dims=[32, 64, 128, 256],
            patch_kernel_size=[3, 3, 3, 3],
            patch_stride=[2, 2, 2, 2],
            patch_padding=[1, 1, 1, 1],
            mlp_ratios=[4, 4, 4, 4],
            num_heads=[1, 2, 4, 8],
            depths=[2, 2, 2, 2],
            decoder_head_embedding_dim=256,
            num_classes=num_classes,
            decoder_dropout=0.0,
        )
        
        if device == 'cuda':
            model3 = model3.cuda()
        model3.eval()
        
        dummy_input3 = torch.randn(input_shape).to(device)
        
        with torch.no_grad():
            output3 = model3(dummy_input3)
        
        flops3 = FlopCountAnalysis(model3, dummy_input3)
        gflops3 = flops3.total() / 1e9
        
        # 测量显存
        print(f"  测量显存 (warmup=3, runs=5)...")
        memory_info3 = measure_model_memory(model3, dummy_input3, device)
        print(f"  峰值显存: {memory_info3['peak_mb']:.2f} MB")
        
        models_info.append({
            'name': 'SegFormer3D',
            'trainable_params': paper_trainable_params['SegFormer3D'],
            'gflops': gflops3,
            'peak_mb': memory_info3['peak_mb'],
            'dice': accuracy_data['SegFormer3D']['dice'],
            'hd95': accuracy_data['SegFormer3D']['hd95']
        })
        
        del model3, dummy_input3
        if device == 'cuda':
            torch.cuda.empty_cache()
        print("✓ 完成\n")
    except Exception as e:
        print(f"✗ 失败: {e}\n")
    
    if len(models_info) == 0:
        print("错误: 没有成功分析任何模型")
        return
    
    # ==================== 打印结果 ====================
    print(f"\n{'='*100}")
    print(f"对比结果")
    print(f"{'='*100}")
    print(f"{'模型':<18} {'可训练参数':>12} {'GFLOPs':>12} {'峰值显存':>12} {'Dice(%)':>12} {'HD95':>12}")
    print(f"{'-'*100}")
    
    for info in models_info:
        params_m = info['trainable_params'] / 1e6
        print(f"{info['name']:<18} {params_m:>11.2f}M {info['gflops']:>12.2f} {info['peak_mb']:>11.2f}MB {info['dice']:>12.2f} {info['hd95']:>12.2f}")
    
    print(f"{'='*100}\n")
    
    return models_info

In [2]:
if __name__ == "__main__":
    compare_models_simple()


模型效率对比 - Synapse数据集
输入尺寸: (1, 1, 64, 128, 128)

设备: cuda

分析 LiteSegFormer3D...


Unsupported operator aten::mul encountered 84 time(s)
Unsupported operator aten::tanh encountered 34 time(s)
Unsupported operator aten::add encountered 94 time(s)
Unsupported operator aten::sub encountered 24 time(s)
Unsupported operator aten::div encountered 32 time(s)
Unsupported operator aten::pow encountered 24 time(s)
Unsupported operator aten::neg encountered 24 time(s)
Unsupported operator aten::exp encountered 24 time(s)
Unsupported operator aten::silu encountered 24 time(s)
Unsupported operator aten::softmax encountered 8 time(s)
Unsupported operator aten::gelu encountered 34 time(s)
Unsupported operator aten::upsample_trilinear3d encountered 4 time(s)
The following submodules of the model were never called during the trace of the graph. They may be unused, or they were accessed by direct calls to .forward() or via other python methods. In the latter case they will have zeros for statistics, though their statistics will still contribute to their parent calling module.
litesegf

  测量显存 (warmup=3, runs=5)...
  峰值显存: 1395.49 MB
✓ 完成

分析 UNETR...


/home/zhangshunyao/miniconda3/envs/unetr_pp/lib/python3.8/site-packages/einops/einops.py:204: UserWarning: __floordiv__ is deprecated, and its behavior will change in a future version of pytorch. It currently rounds toward 0 (like the 'trunc' function NOT 'floor'). This results in incorrect rounding for negative values. To keep the current behavior, use torch.div(a, b, rounding_mode='trunc'), or for actual floor division, use torch.div(a, b, rounding_mode='floor').
  inferred_length: int = length // known_product
Unsupported operator aten::div encountered 89 time(s)
Unsupported operator aten::mul encountered 87 time(s)
Unsupported operator aten::mul_ encountered 14 time(s)
Unsupported operator aten::add encountered 25 time(s)
Unsupported operator aten::softmax encountered 12 time(s)
Unsupported operator aten::gelu encountered 12 time(s)
Unsupported operator aten::leaky_relu_ encountered 10 time(s)
Unsupported operator aten::add_ encountered 5 time(s)


  测量显存 (warmup=3, runs=5)...
  峰值显存: 960.79 MB
✓ 完成

分析 SegFormer3D...


Unsupported operator aten::mul encountered 8 time(s)
Unsupported operator aten::softmax encountered 8 time(s)
Unsupported operator aten::add encountered 16 time(s)
Unsupported operator aten::gelu encountered 8 time(s)
Unsupported operator aten::upsample_trilinear3d encountered 4 time(s)


  测量显存 (warmup=3, runs=5)...
  峰值显存: 2282.93 MB
✓ 完成


对比结果
模型                        可训练参数       GFLOPs         峰值显存      Dice(%)         HD95
----------------------------------------------------------------------------------------------------
LiteSegFormer3D           1.73M       127.53     1395.49MB        84.91         9.41
UNETR                    92.49M        75.69      960.79MB        79.56        22.97
SegFormer3D               4.51M        50.97     2282.93MB        82.15        20.00

